In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
data_path = Path("final_dataset.parquet")
data = pd.read_parquet(data_path)

## Data Split
 

In [2]:
for col in ["news_date_ny", "entry_date", "exit_date"]:
    data[col] = pd.to_datetime(data[col])

in_period = (data["news_date_ny"].ge("2020-01-01") & data["news_date_ny"].lt("2025-01-01"))
has_target = data["excess_return_5d"].notna()
eligible = data.loc[in_period & has_target].copy()
train_mask = eligible["news_date_ny"].lt("2023-01-01")
val_mask = (eligible["news_date_ny"].ge("2023-01-01") & eligible["news_date_ny"].lt("2024-01-01"))
test_mask = eligible["news_date_ny"].ge("2024-01-01")
train = eligible.loc[train_mask & eligible["exit_date"].lt("2023-01-01")].copy()
validation = eligible.loc[val_mask & eligible["exit_date"].lt("2024-01-01")].copy()
test = eligible.loc[test_mask].copy()
split_dir = data_path.parent / "splits"
split_dir.mkdir(parents=True, exist_ok=True)
summary = []
for name, frame in [("train", train),("validation", validation),("test", test),]:
    frame.to_parquet(split_dir / f"{name}_unprocessed.parquet",index=False,)
    summary.append({
        "split": name,
        "rows": len(frame),
        "stocks": frame["PERMNO"].nunique(),
        "first_news_date": frame["news_date_ny"].min(),
        "last_news_date": frame["news_date_ny"].max(),
        "last_exit_date": frame["exit_date"].max(),})
print("News outside 2020-2024:", (~in_period).sum())
print("Missing targets within period:", (in_period & ~has_target).sum())
print("Training rows purged:", train_mask.sum() - len(train))
print("Validation rows purged:", val_mask.sum() - len(validation))
display(pd.DataFrame(summary))

News outside 2020-2024: 5
Missing targets within period: 441
Training rows purged: 308
Validation rows purged: 751


,split,rows,stocks,first_news_date,last_news_date,last_exit_date
0,train,258946,388,2020-01-01,2022-12-22,2022-12-30
1,validation,89502,390,2023-01-02,2023-12-21,2023-12-29
2,test,100380,397,2024-01-01,2024-12-23,2024-12-31


## Imputation and Scaling

In [3]:
missing_pct = train.isna().mean().mul(100).sort_values(ascending=False)
display(missing_pct[missing_pct > 0].rename("missing_percent").to_frame())

,missing_percent
DlyRetMissFlg,99.997297
leverage,4.919559
operating_profitability,0.937647
distance_from_annual_high,0.164127
beta_spy_252d,0.163355
idiosyncratic_volatility_252d,0.163355
sales_growth,0.138639
momentum_12_2,0.134391
asset_growth,0.093070
return_on_assets,0.093070


In [4]:
# Remove the column consistently from all three splits
train = train.drop(columns=["DlyRetMissFlg"])
validation = validation.drop(columns=["DlyRetMissFlg"])
test = test.drop(columns=["DlyRetMissFlg"])


In [5]:
stock_features = [
    "log_market_cap", "book_to_market", "earnings_to_price",
    "sales_to_price", "dividend_yield_12m", "leverage",
    "return_on_assets", "operating_profitability",
    "asset_growth", "sales_growth",
    "return_1d", "return_5d", "return_21d", "momentum_12_2",
    "volatility_20d", "volatility_60d",
    "beta_spy_252d", "idiosyncratic_volatility_252d",
    "share_turnover_20d", "abnormal_volume",
    "log_dollar_volume_20d", "illiquidity_20d",
    "distance_from_annual_high",
    "market_return_5d", "market_volatility_20d", "relative_spread"
]

event_features = [
    "earnings_beat", "earnings_miss",
    "revenue_beat", "revenue_miss", "earnings_guidance",
    "dividend_changes", "share_repurchases", "equity_issuance",
    "debt_issuance", "credit_rating_changes", "default_bankruptcy",
    "mergers_acquisitions", "asset_purchases_sales",
    "analyst_recommendation_changes", "analyst_price_target_changes",
    "management_changes", "insider_transactions", "lawsuits",
    "regulatory_actions", "product_launches", "product_recalls",
    "partnerships_contracts"
]
news_features = [
    "relevance", "event_sentiment_score", "event_relevance",
    "event_similarity_days", "is_fact", "is_forecast", "is_opinion","novel_1day", "novel_7days"
] + event_features + [f"{event}_sentiment" for event in event_features]
all_features = stock_features + news_features
target = "excess_return_5d"

### Imputation by median

Several stock features can be skewed or contain extreme values, such as leverage, trading volume, and valuation ratios. The median is less affected by those extremes than the mean.

In [6]:
X_train = train[all_features].copy()
X_val = validation[all_features].copy()
X_test = test[all_features].copy()
y_train = train[target].copy()
y_val = validation[target].copy()
y_test = test[target].copy()

train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)
print("Training missing values:", X_train.isna().sum().sum())
print("Validation missing values:", X_val.isna().sum().sum())
print("Test missing values:", X_test.isna().sum().sum())

Training missing values: 0
Validation missing values: 0
Test missing values: 0


In [7]:
from sklearn.preprocessing import StandardScaler
# scaled by mean and SD
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [8]:

for name, X, y in [
    ("Train", X_train_scaled, y_train),("Validation", X_val_scaled, y_val),("Test", X_test_scaled, y_test),]:
    print(
        name,"X shape:", X.shape,"y shape:", y.shape,"All finite:", np.isfinite(X).all() and np.isfinite(y).all())

Train X shape: (258946, 79) y shape: (258946,) All finite: True
Validation X shape: (89502, 79) y shape: (89502,) All finite: True
Test X shape: (100380, 79) y shape: (100380,) All finite: True


## Baseline Model

In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict the training target mean for every validation row
baseline_pred = np.full(len(y_val), y_train.mean())
baseline_results = pd.Series({
    "Validation RMSE": np.sqrt(mean_squared_error(y_val, baseline_pred)),
    "Validation MAE": mean_absolute_error(y_val, baseline_pred)})
display(baseline_results)

Validation RMSE    0.034158
Validation MAE     0.024462
dtype: float64

## Autocorrelation Diagnostics

In [ ]:
window_keys = ["PERMNO", "entry_date", "exit_date"]
target = "excess_return_5d"
target_counts = train.groupby(window_keys)[target].nunique()
if target_counts.gt(1).any():
    raise ValueError("Some identical stock/return windows have different targets.")
acf_data = (
    train[window_keys + [target]].dropna(subset=window_keys + [target]).drop_duplicates(subset=window_keys)
    .sort_values(window_keys).copy())

print("Original training rows:", len(train))
print("Unique stock/return windows:", len(acf_data))
print("Repeated windows removed:", len(train) - len(acf_data))

Original training rows: 258946
Unique stock/return windows: 71287
Repeated windows removed: 187659


In [ ]:
lags = [1, 2, 5, 10]
results = []
for permno, group in acf_data.groupby("PERMNO", sort=False):
    returns = group[target].reset_index(drop=True)
    if len(returns) < 50 or returns.nunique() < 2:
        continue
    row = {"PERMNO": permno,"n_windows": len(returns)}
    for lag in lags:
        row[f"acf_lag_{lag}"] = returns.autocorr(lag=lag)
    results.append(row)

acf_results = pd.DataFrame(
    results,columns=["PERMNO", "n_windows"] + [f"acf_lag_{lag}" for lag in lags])
acf_columns = [f"acf_lag_{lag}" for lag in lags]
acf_summary = acf_results[acf_columns].agg(["count", "mean", "median", "min", "max"]).T
display(acf_summary)

# Stocks with the largest absolute lag-1 autocorrelation
display(
    acf_results.assign(abs_acf_lag_1=lambda x: x["acf_lag_1"].abs())
    .sort_values("abs_acf_lag_1", ascending=False).drop(columns="abs_acf_lag_1").head(10))

,count,mean,median,min,max
acf_lag_1,384.0,0.363947,0.352073,-0.157719,0.737915
acf_lag_2,384.0,0.088536,0.068876,-0.372859,0.509077
acf_lag_5,384.0,-0.013333,-0.017712,-0.308619,0.356720
acf_lag_10,384.0,-0.022908,-0.020712,-0.304968,0.339224


,PERMNO,n_windows,acf_lag_1,acf_lag_2,acf_lag_5,acf_lag_10
44,14542,670,0.737915,0.474090,-0.041885,-0.047218
293,84788,626,0.735756,0.449594,-0.020753,-0.003598
357,90319,670,0.731624,0.462568,-0.055346,-0.040362
355,90215,557,0.725879,0.388575,-0.102982,-0.054961
45,14593,532,0.720063,0.425205,-0.001873,-0.008494
27,13407,589,0.717938,0.413238,-0.108248,0.022294
72,19561,390,0.708509,0.283132,-0.125538,-0.082252
233,76226,332,0.705838,0.509077,0.138472,0.052656
62,17478,698,0.705220,0.455832,-0.080932,0.022038
1,10107,638,0.701094,0.456815,-0.066529,-0.098999


Within the training sample, the mean autocorrelation of five-day excess returns across 384 stocks was 0.364 at lag 1 and 0.089 at lag 2, declining to approximately zero at lags 5 and 10. Lags refer to consecutive available return windows within each stock. The short-lag dependence is consistent with overlapping five-day return windows, although this diagnostic does not isolate its cause or test statistical significance. Repeated stock/return windows were excluded from this calculation.

## Elastic Net

In [12]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error

alphas = np.logspace(-6, -2, 9)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
tuning_results = []
best_model = None
best_val_rmse = np.inf

for alpha in alphas:
    for l1_ratio in l1_ratios:
        model = ElasticNet(
            alpha=alpha,l1_ratio=l1_ratio,max_iter=20000,tol=1e-5,selection="cyclic")
        model.fit(X_train_scaled, y_train)
        val_pred = model.predict(X_val_scaled)
        val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        val_mae = mean_absolute_error(y_val, val_pred)
        tuning_results.append({
            "alpha": alpha,"l1_ratio": l1_ratio, "Validation RMSE": val_rmse,"Validation MAE": val_mae,
            "Nonzero coefficients": np.count_nonzero(model.coef_),"Iterations": model.n_iter_})
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_model = model

tuning_results = pd.DataFrame(tuning_results).sort_values("Validation RMSE").reset_index(drop=True)
display(tuning_results.head(10))
print("Best alpha:", best_model.alpha)
print("Best l1_ratio:", best_model.l1_ratio)

C:\Users\Zouuuuu\anaconda3\envs\mqf\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.588054e-02, tolerance: 5.220e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\Zouuuuu\anaconda3\envs\mqf\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.702340e-02, tolerance: 5.220e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\Zouuuuu\anaconda3\envs\mqf\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularis

,alpha,l1_ratio,Validation RMSE,Validation MAE,Nonzero coefficients,Iterations
0,0.010000,0.3,0.034158,0.024462,0,0
1,0.010000,0.5,0.034158,0.024462,0,0
2,0.010000,0.7,0.034158,0.024462,0,0
3,0.010000,0.9,0.034158,0.024462,0,0
4,0.003162,0.9,0.034158,0.024462,0,0
5,0.003162,0.7,0.034161,0.024462,1,2
6,0.003162,0.5,0.034172,0.024469,2,3
7,0.010000,0.1,0.034187,0.024487,6,7
8,0.003162,0.3,0.034189,0.024491,6,6
9,0.001000,0.9,0.034192,0.024495,6,7


Best alpha: 0.0031622776601683794
Best l1_ratio: 0.9


In [14]:
# Candidates that reached max_iter and may not have converged
needs_check = tuning_results[tuning_results["Iterations"] >= 20000]
display(needs_check)

,alpha,l1_ratio,Validation RMSE,Validation MAE,Nonzero coefficients,Iterations
40,0.000001,0.7,0.034577,0.0248,72,20000
41,0.000001,0.5,0.034577,0.0248,72,20000
42,0.000003,0.1,0.034578,0.0248,74,20000
43,0.000001,0.3,0.034578,0.0248,73,20000
44,0.000001,0.1,0.034578,0.0248,77,20000


Five candidates reached the iteration limit. So Rerun those with higher iteration limit.

In [15]:
import warnings
from sklearn.exceptions import ConvergenceWarning

retry_results = []
for idx, row in needs_check.iterrows():
    model = ElasticNet(
        alpha=float(row["alpha"]),l1_ratio=float(row["l1_ratio"]),
        max_iter=100000,tol=1e-5,precompute=True,selection="random",random_state=42)

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X_train_scaled, y_train)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    pred = model.predict(X_val_scaled)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)
    retry_results.append({
        "alpha": model.alpha,"l1_ratio": model.l1_ratio,"Validation RMSE": rmse,
        "Validation MAE": mae,"Iterations": model.n_iter_,"Converged": converged})
    tuning_results.loc[idx, 
        ["Validation RMSE","Validation MAE","Nonzero coefficients","Iterations"]
        ] = [rmse, mae, np.count_nonzero(model.coef_), model.n_iter_]
    if converged and rmse < best_val_rmse:
        best_val_rmse = rmse
        best_model = model
    print(
        f"alpha={model.alpha:.2e}, "
        f"l1_ratio={model.l1_ratio:.1f}: "
        f"converged={converged}, RMSE={rmse:.8f}",
        flush=True)
retry_results = pd.DataFrame(retry_results)
display(retry_results)

alpha=1.00e-06, l1_ratio=0.7: converged=True, RMSE=0.03457698
alpha=1.00e-06, l1_ratio=0.5: converged=True, RMSE=0.03457732
alpha=3.16e-06, l1_ratio=0.1: converged=True, RMSE=0.03457761
alpha=1.00e-06, l1_ratio=0.3: converged=True, RMSE=0.03457764
alpha=1.00e-06, l1_ratio=0.1: converged=True, RMSE=0.03457793


,alpha,l1_ratio,Validation RMSE,Validation MAE,Iterations,Converged
0,0.000001,0.7,0.034577,0.0248,3427,True
1,0.000001,0.5,0.034577,0.0248,4726,True
2,0.000003,0.1,0.034578,0.0248,7434,True
3,0.000001,0.3,0.034578,0.0248,7718,True
4,0.000001,0.1,0.034578,0.0248,32246,True


Elastic Net hyperparameters were selected using 2023 validation RMSE. The selected model shrank all feature coefficients to zero, producing the training-sample mean prediction and matching the baseline. Candidates that initially failed to converge were refitted successfully and did not improve validation performance

In [16]:
# Test predictions
elastic_test_pred = best_model.predict(X_test_scaled)
baseline_test_pred = np.full(len(y_test), y_train.mean())

# Compare test performance
test_results = pd.DataFrame({
    "Baseline": {"Test RMSE": np.sqrt(mean_squared_error(y_test, baseline_test_pred)),
        "Test MAE": mean_absolute_error(y_test, baseline_test_pred)},
    "Elastic Net": {
        "Test RMSE": np.sqrt(mean_squared_error(y_test, elastic_test_pred)),
        "Test MAE": mean_absolute_error(y_test, elastic_test_pred)}}).T

baseline_test_rmse = test_results.loc["Baseline", "Test RMSE"]
test_results["RMSE improvement (%)"] = (
    100 * (baseline_test_rmse - test_results["Test RMSE"]) / baseline_test_rmse)
display(test_results.style.format({"Test RMSE": "{:.8f}","Test MAE": "{:.8f}","RMSE improvement (%)": "{:.4f}"}))

,Test RMSE,Test MAE,RMSE improvement (%)
Baseline,0.03706790,0.02666880,0.0000
Elastic Net,0.03706790,0.02666880,0.0000


Elastic Net selected using 2023 validation RMSE produced zero nonzero feature coefficients. On the 2024 test set, it matched the training-mean baseline, with RMSE of 0.03707 and MAE of 0.02667. Thus, the selected specification provided no predictive improvement over the baseline.